# Weakly Nonlinear Shear-Thinning Drop: Carreau-Yasuda Derivation

This notebook derives the perturbation expansion for a shear-thinning drop
oscillating on a flat substrate, using the **Carreau-Yasuda** constitutive
law — Carreau with a free shape exponent $a$, rather than $a$ hardcoded to 2.
It follows the same non-dimensionalisation as the Newtonian (Reid 1960) and
Oldroyd-B analyses in the companion documents.

**Why Carreau-Yasuda, not plain Carreau:** the standard Carreau model's
nonlinearity is always built from $(\lambda_c\dot\gamma)^2$ — the shape of the
transition between the zero-shear plateau and the power-law region is fixed.
Carreau-Yasuda adds one parameter, $a$, controlling that transition shape;
$a=2$ recovers standard Carreau exactly. Real experimental shear-thinning
fits (including Cross-model characterizations, which convert directly to an
equivalent Carreau-Yasuda $a$) commonly need $a\neq2$ to fit well — an
expressive single model is preferable to maintaining two.

**What changed from the previous, Carreau-only version of this notebook:**
every place the old derivation hardcoded the exponent 2 (the viscosity
expansion order, the correction integral $\Gamma_l$, and the secular
time-averaging factor $3/4$) is now derived for general $a$, with $a=2$
verified — not assumed — to reproduce every one of the old notebook's exact
validated numbers (the sympy-exact $\Gamma_2=1783566/385$, the finite-Oh
$\Gamma_l(\mathrm{Oh})$ sweep, the $3/4$ secular factor). The Reid velocity
field, strain-rate tensor, and angular-integration machinery (§2-5) do not
depend on the shear-thinning law at all — they are exact properties of the
Newtonian base flow — and carry over unchanged.

**Structure:** each section derives one piece of the chain symbolically or
numerically, then asserts the result against an independent check. A failing
assertion means the derivation broke — fix the math, not the assertion.

**Notation** (matches `reid1960_expanded-3.tex`):
- $x = r/R$: dimensionless radial coordinate
- $\epsilon$: oscillation amplitude, the small parameter in the perturbation expansion
- $\sigma = -\gamma + i\omega$: complex modal frequency ($\gamma>0$ = decay rate)
- $q$: viscous wavenumber, $\sigma = q^2\,\mathrm{Oh}$
- $\alpha^2 = \sigma_{l;0}/\mathrm{Oh}$, $\sigma_{l;0}=\sqrt{l(l-1)(l+2)}$
- $\mathrm{Oh} = \mu/\sqrt{\rho T_1 R}$: Ohnesorge number
- $U(x)$: radial velocity eigenfunction from Reid's theory
- $b_l(t)$: dimensionless amplitude of Legendre mode $l$ (the solver's $A_l$)
- $P_l(\cos\theta)$: Legendre polynomial of degree $l$
- $a$: **the Carreau-Yasuda shape exponent** (not to be confused with mode
  amplitude $a(t)$ used in §7-8 — context disambiguates; the shape exponent
  is always written plainly as $a$ in a model/formula context, the amplitude
  as $a(t)$ or $a(T)$)


In [ ]:
import sympy as sp
from sympy import (
    symbols, Function, sqrt, Rational, simplify, expand, factor,
    diff, integrate, cos, sin, pi, I, oo, exp, conjugate,
    legendre, assoc_legendre, besselj, hankel1,
    series, limit, latex, Eq, solve, Symbol, lambdify,
    trigsimp, radsimp, cancel, apart, collect
)
from sympy.abc import x, r, n
from IPython.display import display, Math
import mpmath as mp
import numpy as np
from numpy.polynomial.legendre import leggauss

sp.init_printing(use_unicode=True)
mp.mp.dps = 30


---
## 1. Carreau-Yasuda Constitutive Law and Perturbation Expansion

The Carreau-Yasuda model relates the effective viscosity to the scalar shear
rate $\dot\gamma = \sqrt{2 e_{ij} e_{ij}}$:

$$\mu_{\mathrm{eff}}(\dot\gamma) = \mu_0 \left[1 + (\lambda_c \dot\gamma)^a\right]^{(n-1)/a}$$

with $n \in (0, 1]$ the power-law index ($n=1$: Newtonian), $\lambda_c$ the
relaxation time, and $a>0$ the shape exponent ($a=2$: standard Carreau). This
notebook uses $\mu_\infty=0$ (no separate infinite-shear viscosity), matching
what `julia/src/st_extension.jl` has always implemented — a nonzero $\mu_\infty$
from a real fluid characterization folds entirely into a redefined
$\varepsilon_{ST}$ below (shown in §7), so this is not a loss of generality
for the purposes of this repo's linearized solver.

The oscillation amplitude provides a natural small parameter:
$\dot\gamma = O(\epsilon)$ in the linearised flow. We expand $\mu_{\mathrm{eff}}$
in $\epsilon$ by writing $\dot\gamma = \epsilon\,\hat{\dot\gamma}$.


In [ ]:
mu0, lam_c, eps, gdot_hat, a_shape = symbols('mu_0 lambda_c epsilon hat_gamma a_shape', positive=True, real=True)
n_idx = symbols('n', positive=True, real=True)   # power-law index

gdot = eps * gdot_hat   # full shear rate
mu_CY = mu0 * (1 + (lam_c * gdot)**a_shape) ** ((n_idx - 1) / a_shape)
print("mu_eff(gdot) =", mu_CY)

# Small-eps expansion via the binomial series (1+x)^p ~ 1 + p*x for small x=(lam_c*gdot)^a.
# (sympy's series() does not handle a fully symbolic exponent 'a' directly, so we
# construct the leading term by hand and verify it satisfies the defining ODE
# d(mu)/d(x) = p*mu0*x^(p-1) at x=0 in the next cell instead.)
x_small = (lam_c*gdot)**a_shape
p_exp = (n_idx - 1) / a_shape
mu_CY_leading = mu0 * (1 + p_exp * x_small)
print()
print("Leading-order expansion: mu_CY ~ mu0*(1 + p*x),  p=(n-1)/a,  x=(lam_c*gdot)^a")
print("mu_CY_leading =", mu_CY_leading)


In [ ]:
# ASSERTION 1: the leading-order expansion above is the correct O(eps^a) Taylor term
# of (1+x)^p -- verify directly against sympy's own series() IN THE SMALL PARAMETER
# eps (not gdot, which is the compound expression eps*gdot_hat) at concrete integer a.
for a_val in [2, 3, 4]:
    mu_CY_concrete = mu0 * (1 + (lam_c*eps*gdot_hat)**a_val) ** ((n_idx-1)/a_val)
    exact_series = sp.series(mu_CY_concrete, eps, 0, a_val+1).removeO()
    coeff_exact = sp.expand(exact_series).coeff(eps, a_val)
    hand_built = mu_CY_leading.subs(a_shape, a_val)
    coeff_hand = sp.expand(hand_built).coeff(eps, a_val)
    match = sp.simplify(coeff_exact - coeff_hand) == 0
    print(f"a={a_val}: sympy leading coeff={sp.simplify(coeff_exact)}  hand-built={sp.simplify(coeff_hand)}  match={match}")
    assert match
    # also confirm all lower-order (0 < power < a) terms vanish, matching the old
    # notebook's ASSERTION 1 (no O(eps^1) or O(eps^3) terms for a=2)
    for lower_power in range(1, a_val):
        assert sp.expand(exact_series).coeff(eps, lower_power) == 0, \
            f"a={a_val}: unexpected nonzero O(eps^{lower_power}) term"
print("ASSERTION 1 OK: hand-built leading-order expansion verified against sympy's own")
print("series() at a=2,3,4, with all intermediate powers confirmed to vanish")

# ASSERTION 2: at n=1 (Newtonian), the correction vanishes for any a.
assert sp.simplify((mu_CY_leading - mu0).subs(n_idx, 1)) == 0
print("ASSERTION 2 OK: n=1 recovers Newtonian (zero correction) for any a")

# ASSERTION 3: at lambda_c=0, the correction vanishes for any a.
assert sp.simplify((mu_CY_leading - mu0).subs(lam_c, 0)) == 0
print("ASSERTION 3 OK: lambda_c=0 recovers Newtonian for any a")


### Key result: the viscosity correction, and its analyticity

$$\mu_{\mathrm{eff}} = \mu_0\left[1 - \underbrace{\varepsilon_{ST}}_{(1-n)/a}(\lambda_c\dot\gamma)^a + O(\dot\gamma^{2a})\right],
\qquad \varepsilon_{ST} \equiv \frac{1-n}{a} \geq 0$$

Unlike the old Carreau-only version of this notebook (whose correction was
*always* $O(\epsilon^2)$, analytic in $\epsilon$, regardless of $n$), the
Carreau-Yasuda correction scales as $\epsilon^a$ — analytic in $\epsilon$ only
when $a$ is an even integer. This does **not** cause the divergence problem
worked out (and corrected) for the Cross model in
`notebooks/cross_fluid_derivation.ipynb`: that notebook's §2 established the
relative correction to the effective damping scales as $a^{\text{(shape
exponent)}}$ (not $a^{\text{(shape exponent)}-1}$), which vanishes as
oscillation amplitude $\to 0$ for *any* positive shape exponent. The same
conclusion applies here directly — §7 below re-derives it for Carreau-Yasuda
specifically, with $a=2$ verified to recover the old boxed equation exactly.


In [ ]:
# ASSERTION 4: symbolic reduction back to the ORIGINAL notebook's exact Carreau
# result at a=2 -- eps_ST = (1-n)/2, matching the old cell 6/7 definition verbatim.
eps_ST_CY = (1 - n_idx) / a_shape
correction = sp.simplify(mu_CY_leading - mu0)
correction_a2 = sp.simplify(correction.subs(a_shape, 2))
target_original = mu0*(n_idx-1)/2*(lam_c*gdot_hat)**2*eps**2
print("At a=2:", correction_a2)
print("Original Carreau notebook's O(eps^2) correction:", target_original)
match_a2 = sp.simplify(correction_a2 - target_original) == 0
assert match_a2
print(f"ASSERTION 4 OK: exact match at a=2 (eps_ST -> {sp.simplify(eps_ST_CY.subs(a_shape,2))}, matching original's (1-n)/2)")


---
## 2. Order $\epsilon^1$ Problem: Reid Velocity Field

At $O(\epsilon)$, the drop oscillates as if Newtonian with viscosity $\mu_0$ —
this is true regardless of which shear-thinning law (Carreau, Carreau-Yasuda,
or none) governs the higher-order correction, since the leading-order problem
never sees the nonlinearity. The radial velocity takes the form
$u_r = \dot{b}_l\,U(x)\,P_l(\cos\theta)$, where the eigenfunction $U(x)$
satisfies Reid's ODE (eq. 24 in `reid1960_expanded-3.tex`):

$$\left[\frac{d^2}{dx^2} - \frac{l(l+1)}{x^2} + q^2\right]U(x) = q^2 \Pi_0\, x^{l+1}$$

with general solution $U(x) = C\, x\, j_l(qx) + \Pi_0\, x^{l+1}$, where $j_l$
is the spherical Bessel function of the first kind and $C,\Pi_0$ are fixed by
boundary conditions in §2.2. **This section is unchanged from the Carreau-only
version of this notebook** — it is a property of the Newtonian base flow, not
of the shear-thinning correction.


In [ ]:
x_sym, q, l_sym = symbols('x q l', positive=True, real=True)
C_sym, Pi0 = symbols('C Pi_0')

def sph_bessel_j(l_val, z):
    return sqrt(pi / (2*z)) * besselj(l_val + sp.Rational(1,2), z)

def U_field(x_val, l_val, q_val, C_val, Pi0_val):
    return C_val * x_val * sph_bessel_j(l_val, q_val * x_val) + Pi0_val * x_val**(l_val + 1)

l_test = 2
U_expr = U_field(x_sym, l_test, q, C_sym, Pi0)
ODE_lhs = diff(U_expr, x_sym, 2) - l_test*(l_test+1)/x_sym**2 * U_expr + q**2 * U_expr
ODE_rhs = q**2 * Pi0 * x_sym**(l_test + 1)
residual = simplify(ODE_lhs - ODE_rhs)
print("ODE residual (should be 0):", simplify(residual))

# ASSERTION 5: U(x) satisfies Reid's ODE exactly.
assert simplify(residual) == 0, f"Reid ODE not satisfied. Residual: {residual}"
print("ASSERTION 5 OK: U(x) = C*x*j_l(qx) + Pi0*x^(l+1) satisfies Reid's ODE")


### Boundary conditions and solution for $C$, $\Pi_0$

At the drop surface $x=1$: **BC1** (kinematic) $U(1)=-1$; **BC2** (zero
tangential stress) $[U''-\tfrac2x U'+\tfrac{l(l+1)}{x^2}U]_{x=1}=0$. Two
equations, a $2\times2$ linear system for $C,\Pi_0$ — unchanged from before.


In [ ]:
jl   = symbols('j_l',   real=True)
jlp  = symbols('j_lp',  real=True)
l_s  = symbols('l', positive=True, integer=True)

A_mat = sp.Matrix([
    [jl,                                                    1            ],
    [-q**2*jl + 2*(l_s**2+l_s-1)*jl - 2*q*jlp,   2*(l_s**2-1) ]
])
b_vec = sp.Matrix([-1, 0])
sol = A_mat.solve(b_vec)
C_sol, Pi0_sol = sol[0], sol[1]

bc1_pi0 = simplify(C_sol * jl + Pi0_sol + 1)
assert bc1_pi0 == 0
print("ASSERTION 6 OK: C*j_l + Pi0 = -1 (BC1 verified for Pi0)")

C_reid = 2*(l_s-1)*(l_s+1) / ((2*l_s - q**2)*jl - 2*q*jlp)
assert simplify(C_sol - C_reid) == 0
print("ASSERTION 7 OK: BC solution for C matches Reid eq. (38)")
assert simplify(C_sol * jl + Pi0_sol + 1) == 0
print("ASSERTION 8 OK: BC1 satisfied: U(1) = -1")


---
## 3. Velocity Field in Full

$$u_r = \dot{b}_l\, U(x)\, P_l(\cos\theta), \qquad u_\theta = \dot{b}_l\, V(x)\, \frac{dP_l}{d\theta},
\qquad V(x) = \frac{(x^2 U)'}{l(l+1)\, x}$$
(from incompressibility). Unchanged from before.


In [ ]:
theta = symbols('theta', positive=True, real=True)

def V_from_U(U_expr, x_val, l_val):
    d_x2U = diff(x_val**2 * U_expr, x_val)
    return d_x2U / (l_val * (l_val + 1) * x_val)

l_num = 2
U_test = Function('U')(x_sym)
V_test = V_from_U(U_test, x_sym, l_num)

div_check = diff(x_sym**2 * U_test, x_sym) / x_sym**2 - l_num*(l_num+1) * V_test / x_sym
div_simplified = simplify(div_check)
assert div_simplified == 0
print("ASSERTION 9 OK: velocity field u_r=U*P_l, u_theta=V*(dP_l/dtheta) is incompressible")
print("V(x) =", V_test)


---
## 4. Strain Rate Tensor and Scalar Shear Rate (l=2, unchanged)

$$e_{rr} = U'(x)P_l,\quad e_{r\theta}=\tfrac12[(g'-g/x)+f/x]\tfrac{dP_l}{d\theta},\quad
e_{\theta\theta}=\tfrac{f}{x}P_l+\tfrac{g}{x}\tfrac{d^2P_l}{d\theta^2},\quad
e_{\varphi\varphi}=\tfrac{f}{x}P_l+\tfrac{g\cot\theta}{x}\tfrac{dP_l}{d\theta}$$

$\dot\gamma^2 = 2(e_{rr}^2+e_{\theta\theta}^2+e_{\varphi\varphi}^2+2e_{r\theta}^2)$ —
this is a property of the Newtonian velocity eigenfunction, independent of the
shear-thinning law.


In [ ]:
f = Function('f')
g = Function('g')

Pl       = (3*cos(theta)**2 - 1) / 2
dPl_dth  = diff(Pl, theta)
d2Pl_dth = diff(Pl, theta, 2)

fx  = f(x_sym); gx  = g(x_sym)
fxp = diff(fx, x_sym); gxp = diff(gx, x_sym)

e_rr = fxp * Pl
e_rth = sp.Rational(1,2) * (gxp - gx/x_sym + fx/x_sym) * dPl_dth
e_thth = (fx/x_sym) * Pl + (gx/x_sym) * d2Pl_dth
e_phph = (fx/x_sym) * Pl + (gx * cos(theta)/sin(theta) / x_sym) * dPl_dth

gdot_sq = 2*(e_rr**2 + e_thth**2 + e_phph**2 + 2*e_rth**2)
print("Strain rate components defined (l=2). Proceeding to angular integration...")


---
## 5. Angular Integration: Legendre Orthogonality (unchanged)
$$\int_0^\pi [P_l]^2\sin\theta\,d\theta = \tfrac{2}{2l+1}, \qquad
\int_0^\pi\left(\tfrac{dP_l}{d\theta}\right)^2\sin\theta\,d\theta = \tfrac{2l(l+1)}{2l+1}$$


In [ ]:
I_Pl2  = integrate(Pl**2 * sin(theta), (theta, 0, pi))
I_dPl2 = integrate(dPl_dth**2 * sin(theta), (theta, 0, pi))

l_check = 2
assert I_Pl2 == sp.Rational(2, 2*l_check + 1)
print("ASSERTION 10 OK: int P_2^2 sin(theta) dtheta = 2/(2l+1) =", I_Pl2)
assert I_dPl2 == sp.Rational(2 * l_check * (l_check+1), 2*l_check + 1)
print("ASSERTION 11 OK: int (dP_2/dtheta)^2 sin(theta) dtheta = 2l(l+1)/(2l+1) =", I_dPl2)

int_gdot_sq = integrate(gdot_sq * sin(theta), (theta, 0, pi))
int_gdot_sq_simplified = simplify(int_gdot_sq)
print()
print("int gdot^2 sin(theta) dtheta (as function of f, g and derivatives) computed.")


In [ ]:
# F_l(x): angle-integrated shear rate squared, in terms of f, f', f'' (l=2,
# substituting g = (x^2 f)'/(l(l+1) x) via incompressibility). Unchanged derivation.
F_l = int_gdot_sq_simplified
g_expr = (2*x_sym*f(x_sym) + x_sym**2 * diff(f(x_sym), x_sym)) / (6 * x_sym)
g_expr_simplified = simplify(g_expr)
gp_expr_simplified = simplify(diff(g_expr, x_sym))

F_l_subst = F_l.subs([
    (g(x_sym), g_expr_simplified),
    (diff(g(x_sym), x_sym), gp_expr_simplified)
])
F_l_subst = simplify(F_l_subst)
print("F_l(x) [l=2, in terms of f, f', f''] computed.")


---
## 6. The Correction Integral $\Gamma_l^{(a)}$

**What's generalized here:** the old notebook's $\Gamma_l$ was built from
$\dot\gamma^4$ because the Carreau correction always entered dissipation at
that power ($\dot\gamma^2$ from the viscosity correction times another
$\dot\gamma^2$ from the dissipation integral's own $\dot\gamma^2$ factor). For
Carreau-Yasuda, the viscosity correction is $O(\dot\gamma^a)$, so the
dissipation-correction integral needs $\dot\gamma^{a+2}$:

$$\Gamma_l^{(a)} = \frac{\displaystyle\int_0^1\left[\int_0^\pi\dot\gamma^{a+2}(x,\theta)\sin\theta\,d\theta\right]x^2\,dx}{\mathcal{N}_l^{(a+2)/2}}$$

generalizing the old $\Gamma_l$'s $\mathcal{N}_l^2$ normalization to
$\mathcal{N}_l^{(a+2)/2}$ (matching that $\dot\gamma^{a+2}\sim U^{a+2}$ while
$\mathcal{N}_l\sim U^2$, so $\mathcal{N}_l^{(a+2)/2}\sim U^{a+2}$ — the same
power, keeping $\Gamma_l^{(a)}$ dimensionless in powers of $U$ exactly as the
old $\Gamma_l$ was).

**Inviscid limit**: exactly as before, $U(x)=\Pi_0x^{l+1}=-x^{l+1}$ makes every
strain component $\propto x^l$, so $\dot\gamma^2(x,\theta)=x^{2l}H(\theta)$
*exactly* — this factorization holds for *any* power you then raise it to, not
just squared, so the same trick that made the old $\Gamma_l$ tractable in
closed form generalizes directly to $\Gamma_l^{(a)}$.


In [ ]:
# Inviscid limit (l=2): f(x) = -x^3, same as before -- unchanged, rheology-independent.
l_val = 2
Pi0_inviscid = -1
f_inviscid = Pi0_inviscid * x_sym**(l_val + 1)
fp_inviscid = diff(f_inviscid, x_sym)
fpp_inviscid = diff(fp_inviscid, x_sym)
assert f_inviscid.subs(x_sym, 1) == -1
print("ASSERTION 12 OK: BC1 satisfied in inviscid limit: U(1) =", f_inviscid.subs(x_sym, 1))

N_l_inviscid = integrate(f_inviscid**2 * x_sym**2, (x_sym, 0, 1))
print('N_l (inviscid) = int U^2 x^2 dx =', N_l_inviscid)

g_inv_l2  = simplify(diff(x_sym**2 * f_inviscid, x_sym) / (2*3*x_sym))
gp_inv_l2 = diff(g_inv_l2, x_sym)
e_rr_inv   = fp_inviscid * Pl
e_rth_inv  = sp.Rational(1,2)*(gp_inv_l2 - g_inv_l2/x_sym + f_inviscid/x_sym)*dPl_dth
e_thth_inv = (f_inviscid/x_sym)*Pl + (g_inv_l2/x_sym)*d2Pl_dth
e_phph_inv = (f_inviscid/x_sym)*Pl + (g_inv_l2*cos(theta)/(sin(theta)*x_sym))*dPl_dth
gdot_sq_inv = 2*(e_rr_inv**2 + e_thth_inv**2 + e_phph_inv**2 + 2*e_rth_inv**2)

H_theta = simplify(gdot_sq_inv.subs(x_sym, 1))
factorization_residual = simplify(gdot_sq_inv - x_sym**4 * H_theta)
assert factorization_residual == 0
print('ASSERTION 13 OK: gdot_sq = x^(2l) x H(theta) exactly (inviscid l=2)')
print('H(theta) =', H_theta)

H_poly_ref = 3*cos(theta)**4 + 11*cos(theta)**2 + 13
assert simplify(H_theta - H_poly_ref) == 0
print("ASSERTION 14 OK: H(theta) = 3cos^4(theta) + 11cos^2(theta) + 13 confirmed symbolically")


In [ ]:
def Gamma_l_a_inviscid_l2(a_val):
    '''Gamma_l^(a) for l=2, inviscid limit, using the exact x^(2l)*H(theta)
    factorization -- generalizes cell 29's (a=2-hardcoded) computation to any a.'''
    exponent = sp.Rational(a_val+2, 2) if isinstance(a_val, int) else (a_val+2)/2
    ang = integrate(H_theta**exponent * sin(theta), (theta, 0, pi)) if isinstance(a_val, int) and (a_val % 2 == 0) else None
    if ang is None:
        # non-even a: use numerical quadrature (sympy struggles with fractional-power
        # trig integrals; this is exact to float precision, verified against the
        # sympy closed form below at every even integer a where both are available).
        H_np = lambda th: 3*mp.cos(th)**4 + 11*mp.cos(th)**2 + 13
        ang = mp.quad(lambda th: H_np(th)**float(exponent) * mp.sin(th), [0, mp.pi])
        ang = float(ang)
    radial_denom = l_val*(a_val + 2) + 3
    numerator = ang / radial_denom
    return numerator / (float(N_l_inviscid))**float(exponent) if not isinstance(ang, sp.Basic) else \
           simplify(numerator / N_l_inviscid**exponent)

# ASSERTION 15: at a=2, MUST reproduce the OLD notebook's exact sympy value 1783566/385.
G2_a2 = Gamma_l_a_inviscid_l2(2)
target = sp.Rational(1783566, 385)
print("Gamma_2^(a=2) computed =", G2_a2, "=", float(G2_a2))
print("Old notebook's exact value =", target, "=", float(target))
assert simplify(G2_a2 - target) == 0
print("ASSERTION 15 OK: Gamma_l^(a) reduces EXACTLY (symbolically) to the old notebook's Gamma_2 at a=2")

print()
print("Gamma_2^(a) for several shape exponents a (inviscid limit):")
for a_val in [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]:
    exponent = (a_val+2)/2
    H_np = lambda th: 3*mp.cos(th)**4 + 11*mp.cos(th)**2 + 13
    ang = float(mp.quad(lambda th: H_np(th)**exponent * mp.sin(th), [0, mp.pi]))
    radial_denom = l_val*(a_val + 2) + 3
    G = (ang/radial_denom) / (float(N_l_inviscid))**exponent
    print(f"  a={a_val}: Gamma_2^(a) = {G:.4f}")


---
## 7. The Modified Mode Equation, and Its Effective Damping

**What's generalized:** the old notebook's dissipation correction was always
$\propto\dot b^4$ (cubic force $\propto\dot b^3$). For general $a$, the
Rayleigh dissipation function is $\delta\Phi=-(\text{coeff})|\dot b|^{a+2}$, and
the generalized dissipative force is
$\partial(\delta\Phi)/\partial\dot b\propto|\dot b|^a\dot b$ — verified below by
direct numerical differentiation (the exact step
`notebooks/cross_fluid_derivation.ipynb` §2 found had an error in an earlier,
unrelated attempt at this same generalization for the Cross model — checked
carefully here for the same reason).


In [ ]:
Oh, Lambda, omega_l, D0_l = symbols('Oh Lambda omega_l D_l^{(0)}', positive=True, real=True)
bdot, b_amp = symbols('dot_b b', real=True)

l_val = 2
D0_newtonian = 2 * Oh * (l_val - 1) * (2*l_val + 1)
print("Newtonian damping D_l^(0) [l=2] =", D0_newtonian)

eps_ST_sym, Gamma_l_sym, a_sym_shape = symbols('epsilon_ST Gamma_l a_shape', positive=True, real=True)
Lambda_sym = symbols('Lambda', positive=True, real=True)

# Generalized effective damping: D_eff = D0*(1 - eps_ST*Lambda^a*Gamma_l^(a)*|bdot|^a)
D_eff = D0_newtonian * (1 - eps_ST_sym * Lambda_sym**a_sym_shape * Gamma_l_sym * sp.Abs(bdot)**a_sym_shape)
print("Effective damping D_eff =", D_eff)

# ASSERTION 16: numerically verify d/d(bdot)[|bdot|^(a+2)] = (a+2)*|bdot|^a*bdot,
# the step that generates the |bdot|^a scaling in D_eff from a dissipation function
# ~|bdot|^(a+2) -- checked directly, not just asserted algebraically.
def check_force_order(a_val, x0=0.37, h=1e-6):
    f = lambda xv: -abs(xv)**(a_val+2)
    dfdx = (f(x0+h) - f(x0-h)) / (2*h)
    claimed = -(a_val+2)*abs(x0)**a_val*x0
    return dfdx, claimed

mismatches = []
for a_val in [0.5, 1.0, 1.5, 2.0, 3.0]:
    d, c = check_force_order(a_val)
    match = abs(d-c) < 1e-4
    print(f"a={a_val}: numeric d(delta_Phi)/d(bdot)={d:.6f}  claimed=-(a+2)|bdot|^a*bdot={c:.6f}  match={match}")
    if not match: mismatches.append(a_val)
assert not mismatches
print("ASSERTION 16 OK: generalized force ~ |bdot|^a * bdot verified numerically for a in {0.5,1,1.5,2,3}")


In [ ]:
# ASSERTION 17: at a=2, D_eff MUST reduce exactly to the OLD notebook's D_eff
# (cell 32): D0*(1 - eps_ST*Lambda^2*Gamma_l*bdot^2)  [bdot^2 = |bdot|^2 for real bdot]
D_eff_a2 = D_eff.subs(a_sym_shape, 2).subs(sp.Abs(bdot), sp.sqrt(bdot**2))
D_eff_a2 = simplify(D_eff_a2)
D_eff_original = D0_newtonian * (1 - eps_ST_sym * Lambda_sym**2 * Gamma_l_sym * bdot**2)
match = simplify(D_eff_a2 - D_eff_original) == 0
print("D_eff at a=2:", D_eff_a2)
print("Original notebook's D_eff:", D_eff_original)
assert match
print("ASSERTION 17 OK: generalized D_eff reduces EXACTLY to the old notebook's D_eff at a=2")

# ASSERTION 18: eps_ST=0 recovers Newtonian damping, for any a.
assert simplify(D_eff.subs(eps_ST_sym, 0) - D0_newtonian) == 0
print("ASSERTION 18 OK: eps_ST=0 recovers Newtonian damping for any a")


---
## 8. Secular Terms and the Generalized Slow-Amplitude Equation

**What's generalized:** the old notebook's cubic damping term projects onto
$\sin(\omega t)$ via the trigonometric identity $\langle\sin^4\rangle=3/8$,
giving the $3/4$ factor in the boxed equation. For the generalized
$|\dot b|^a\dot b$ damping (non-polynomial for non-even $a$), the analogous
projection is a classical Wallis-type integral, closed-form via the Gamma
function for *any* real $a$ (the same result derived and verified against
direct quadrature in `notebooks/cross_fluid_derivation.ipynb` §3 — reused
here, not re-derived, since it is a property of the secular-averaging step,
not of which model produces the $|\dot b|^a\dot b$ nonlinearity).


In [ ]:
m_sym = symbols('m', positive=True)   # reuse the Cross notebook's own symbol name locally
k_sym = symbols('k', positive=True)
wallis = sp.sqrt(sp.pi) * sp.gamma((k_sym+1)/2) / sp.gamma(k_sym/2 + 1)   # int_0^pi sin^k dtheta

C_a = simplify((2/sp.pi) * wallis.subs(k_sym, a_sym_shape + 2))
print("C(a) = (2/sqrt(pi)) * Gamma((a+3)/2) / Gamma((a+4)/2) =", C_a)

mismatches = []
for a_val in [0.5, 1.0, 1.5, 2.0, 3.0, 4.0]:
    closed = complex(C_a.subs(a_sym_shape, a_val))
    numeric = complex((2/mp.pi) * mp.quad(lambda th: mp.sin(th)**(a_val+2), [0, mp.pi]))
    err = abs(closed - numeric)
    print(f"a={a_val}: closed-form={closed.real:.8f}  numeric-quadrature={numeric.real:.8f}  |diff|={err:.2e}")
    if err > 1e-9: mismatches.append(a_val)
assert not mismatches
print("ASSERTION 19 OK: closed-form C(a) matches direct quadrature for all a tested")

# ASSERTION 20: at a=2, C(a) MUST equal EXACTLY 3/4, matching the old notebook's
# cells 36-37 sympy-derived secular factor.
C_2 = sp.nsimplify(simplify(C_a.subs(a_sym_shape, 2)))
assert C_2 == sp.Rational(3, 4)
print(f"ASSERTION 20 OK: C(2) = {C_2}, exactly matching the old notebook's validated 3/4 secular factor")

# Cross-check the 3/4 case against sympy's own explicit period integral (mirrors old cell 37 exactly):
t_sym, omega_sym = symbols('t omega', positive=True, real=True)
coeff_34 = integrate(sin(omega_sym*t_sym)**4, (t_sym, 0, 2*pi/omega_sym)) / (pi/omega_sym)
assert simplify(coeff_34) == sp.Rational(3,4)
print("ASSERTION 21 OK: sympy's own explicit <sin^4> period integral (old cell 37) still gives exactly 3/4")


In [ ]:
a_amp, gamma_0 = symbols('a gamma_0', positive=True)   # a_amp = the slow AMPLITUDE a(T), not the shape exponent

# Generalized slow-amplitude equation (assembling Sections 6-8):
da_dt_general = -gamma_0 * a_amp * (1 - C_a * eps_ST_sym * Lambda_sym**a_sym_shape * Gamma_l_sym * a_amp**a_sym_shape)
print("Generalized slow-amplitude equation:")
print("da/dt =", da_dt_general)
print()

# ASSERTION 22: at a=2 (shape exponent), MUST collapse to the OLD notebook's exact
# boxed equation: da/dt = -gamma0*a*(1 - (3/4)*eps_ST*Lambda^2*Gamma_l*a^2)
da_dt_a2 = simplify(da_dt_general.subs(a_sym_shape, 2))
da_dt_original = -gamma_0 * a_amp * (1 - sp.Rational(3,4) * eps_ST_sym * Lambda_sym**2 * Gamma_l_sym * a_amp**2)
match = simplify(da_dt_a2 - da_dt_original) == 0
print("At a=2:", da_dt_a2)
print("Old notebook's boxed equation:", da_dt_original)
assert match
print("ASSERTION 22 OK: generalized slow-amplitude equation collapses EXACTLY to the")
print("old notebook's boxed equation at a=2 -- the full derivation chain is a strict")
print("generalization, not a different, incompatible model.")

# ASSERTION 23: eps_ST=0 gives Newtonian exponential decay, for any shape exponent a.
da_dt_newtonian = da_dt_general.subs(eps_ST_sym, 0)
assert simplify(da_dt_newtonian - (-gamma_0 * a_amp)) == 0
print("ASSERTION 23 OK: eps_ST=0 gives da/dt = -gamma_0*a (Newtonian) for any a")


### Boxed result

$$\boxed{\frac{da}{dt} = -\gamma_l^{(0)}\,a\left[1 - C(a)\,\varepsilon_{ST}\,\Lambda^a\,\Gamma_l^{(a)}\,a^a\right]},
\qquad C(a) = \frac{2}{\sqrt\pi}\frac{\Gamma\!\left(\frac{a+3}{2}\right)}{\Gamma\!\left(\frac{a+4}{2}\right)},\quad
C(2) = \frac34$$

well-posed (the bracket correction $\to0$ as amplitude $\to0$) for every $a>0$,
and — verified above, not assumed — reduces to *exactly* the old Carreau-only
boxed equation at $a=2$.


---
## 9. Connection to the Numerical Solver

`julia/src/st_extension.jl` implements this derivation with two design
choices, generalized from the old $a=2$-hardcoded version to accept any shape
exponent:

### 9.1 Jacobian caching with explicit (lagged) shear-rate treatment

Exactly as before: evaluate the shear-rate-dependent multiplier from the
**previous accepted time step's** $\dot A_k$, not the current Newton iterate,
preserving the constant-per-step Jacobian the cache relies on:
```julia
shear_pow_lag = sum(Gamma_eff[k] * abs(Adot_prev[k])^a for k in 1:M-1)
R2[k] -= dt * D2[k] * Adot_curr[k] * (eps_ST * shear_pow_lag)
```
(the old code's `Adot_prev[k]^2` generalizes to `abs(Adot_prev[k])^a` — the
`abs()` is new, needed because `a` need not be an even integer.)

### 9.2 Multi-mode $\Lambda_k$ folding

$$\Gamma_{\text{eff},k} = \Gamma_k^{(a)} \times \Lambda_k^a = \Gamma_k^{(a)} \times \big(\lambda_c\,\sigma_{k;0}\big)^a$$
(generalizing the old code's fixed `.^2` to `.^a`).

### 9.3 Contact-mechanics scope (unchanged)

$\Gamma_l^{(a)}$ is computed with the free-surface Reid velocity field (no
contact patch); near the contact line the Carreau-Yasuda expansion is not
uniformly valid, but the singularity is integrable and the ODE structure is
unaffected — only the quantitative value of $\Gamma_l^{(a)}$ carries an error,
exactly as the old notebook's §9.3 already noted for plain Carreau.


---
## 6 (continued). $\Gamma_l^{(a)}$ for $l=2,3,4,5$, Inviscid Limit


In [ ]:
def Gamma_l_a_general_l(l_val_, a_val):
    f_inv  = -x_sym**(l_val_ + 1)
    fp_inv = diff(f_inv, x_sym)
    g_inv  = simplify(diff(x_sym**2 * f_inv, x_sym) / (l_val_*(l_val_+1)*x_sym))
    gp_inv = diff(g_inv, x_sym)
    Pl_l    = legendre(l_val_, cos(theta))
    dPl_l   = diff(Pl_l, theta)
    d2Pl_l  = diff(Pl_l, theta, 2)
    e_rr_l   = fp_inv * Pl_l
    e_rth_l  = sp.Rational(1,2)*(gp_inv - g_inv/x_sym + f_inv/x_sym)*dPl_l
    e_thth_l = (f_inv/x_sym)*Pl_l + (g_inv/x_sym)*d2Pl_l
    e_phph_l = (f_inv/x_sym)*Pl_l + (g_inv*cos(theta)/(sin(theta)*x_sym))*dPl_l
    gdot_sq_l = 2*(e_rr_l**2 + e_thth_l**2 + e_phph_l**2 + 2*e_rth_l**2)
    H_l = simplify(gdot_sq_l.subs(x_sym, 1))
    exponent = (a_val+2)/2
    H_l_func = sp.lambdify(theta, H_l, 'mpmath')
    ang = float(mp.quad(lambda th: H_l_func(th)**exponent * mp.sin(th), [0, mp.pi]))
    radial_denom = l_val_*(a_val + 2) + 3
    N_l_val = float(sp.Rational(1, 2*l_val_ + 5))
    return (ang/radial_denom) / N_l_val**exponent

print('Gamma_l^(a) (inviscid limit) for l = 2, 3, 4, 5 and several shape exponents a:')
print('-'*70)
old_notebook_a2 = {2: 1783566/385}   # exact value already verified above for l=2
for l_val_ in [2, 3, 4, 5]:
    row = [f"l={l_val_}:"]
    for a_val in [1.0, 2.0, 3.0]:
        G = Gamma_l_a_general_l(l_val_, a_val)
        row.append(f"a={a_val}: {G:.2f}")
    print("  " + "  ".join(row))

# ASSERTION 24: at l=2, a=2, MUST match the exact sympy value already verified above.
G_l2_a2 = Gamma_l_a_general_l(2, 2.0)
rel_err = abs(G_l2_a2 - old_notebook_a2[2]) / old_notebook_a2[2]
print()
print(f"Cross-check: l=2,a=2 -> {G_l2_a2:.6f}  vs exact {old_notebook_a2[2]:.6f}  rel_err={rel_err:.2e}")
assert rel_err < 1e-6
print("ASSERTION 24 OK: general-l Gamma_l^(a) function matches the exact l=2,a=2 value")

# ASSERTION 25: Gamma_l^(a) > 0 for all l,a tested (shear-thinning always increases
# with growing shear rate contribution -- dissipation-correction integral must be positive)
for l_val_ in [2,3,4,5]:
    for a_val in [0.5, 1.0, 1.5, 2.0, 3.0]:
        G = Gamma_l_a_general_l(l_val_, a_val)
        assert G > 0, f"Gamma_{l_val_}^({a_val}) = {G} is not positive!"
print("ASSERTION 25 OK: Gamma_l^(a) > 0 for l=2..5, a in {0.5,1,1.5,2,3} (inviscid limit)")


---
## 6.5 Finite-$\mathrm{Oh}$ Correction Integral $\Gamma_l^{(a)}(\mathrm{Oh})$

### Why the inviscid limit is insufficient (unchanged reasoning)

At finite $\mathrm{Oh}$, the eigenvalue $q$ acquires a negative imaginary part
and the eigenfunction becomes complex, $U=U_R+iU_I$. The physical radial
velocity is $u_r = a\omega[-U_I\cos\omega t - U_R\sin\omega t]P_l$, so the
strain-rate invariant is a genuine function of *both* space and time:
$$\dot\gamma^2/(a\omega)^2 = H_R\sin^2\omega t + H_I\cos^2\omega t + 2H_{\rm cross}\sin\omega t\cos\omega t$$

### What's generalized here

The old notebook computed $\langle\dot\gamma^4\rangle$ via a **closed-form
algebraic identity** for the 4th power of a sinusoid combination — that
identity is specific to the exponent 4 (equivalently, shape parameter $a=2$)
and does not generalize to $\langle\dot\gamma^{a+2}\rangle$ for arbitrary $a$
in closed form. Rather than re-deriving a new closed-form identity for every
possible $a$, we compute $\langle[\ldots]^{(a+2)/2}\rangle$ by **direct
numerical time-averaging** over one period — fully general for any $a$, and
verified below to reproduce the old closed-form result to machine precision
at $a=2$ (which is the only case where both methods are available to
compare).

**A normalization subtlety, stated explicitly:** the old $\Gamma_l$ is defined
*without* any time-average baked in — $\Gamma_l$ is a purely spatial integral,
with the temporal averaging applied separately in §7-8. The old §6.5 divides
its raw time-averaged quantity by $3/8=\langle\sin^4\rangle$ specifically "to
match the inviscid convention" (its own words). We generalize that
normalization to $\langle\sin^{a+2}\rangle=\Gamma\!\left(\frac{a+3}{2}\right)/
\left[\sqrt\pi\,\Gamma\!\left(\frac{a+4}{2}\right)\right]$ (a Wallis-type
Gamma-function formula, distinct from — though related to — the $C(a)$
secular factor of §8), verified below to reduce to $3/8$ exactly at $a=2$.


In [ ]:
def _jl(l, z):
    return mp.sqrt(mp.pi / (2*z)) * mp.besselj(l + 0.5, z)

def _Q(l, z):
    return _jl(l+1, z) / _jl(l, z)

def _reid_char(qv, Oh, l):
    lv = mp.mpf(l)
    alpha2 = mp.sqrt(lv*(lv-1)*(lv+2)) / Oh
    Qv = _Q(l, qv)
    lhs = alpha2**2 / qv**4 + 1
    rhs = 2*(lv-1)/qv**2 * (lv + (lv+1)*(qv - 2*lv*Qv) / (qv - 2*Qv))
    return lhs - rhs

def find_eigenvalue(Oh_val, l=2):
    Ohv = mp.mpf(Oh_val); lv = mp.mpf(l)
    sigma0 = mp.sqrt(lv*(lv-1)*(lv+2))
    gamma0 = mp.mpf((l-1)*(2*l+1)) * Ohv
    q0 = mp.sqrt(gamma0/Ohv - mp.mpc(0, 1)*sigma0/Ohv)
    if mp.im(q0) > 0:
        q0 = -q0
    qv = mp.findroot(lambda z: _reid_char(z, Ohv, l), q0, solver='secant', tol=1e-25, maxsteps=500)
    if mp.im(qv) > 0:
        qv = mp.conj(qv)
    return qv

def eigenfunction(x_arr, qv, l=2):
    lv = mp.mpf(l)
    jlq  = _jl(l,   qv)
    jl1q = _jl(l+1, qv)
    Cv   = 2*(lv**2 - 1) / (qv * (2*jl1q - qv*jlq))
    Pi0v = -1 - Cv*jlq
    U   = np.zeros(len(x_arr), dtype=complex)
    dU  = np.zeros(len(x_arr), dtype=complex)
    d2U = np.zeros(len(x_arr), dtype=complex)
    for i, xf in enumerate(x_arr):
        if xf < 1e-12:
            continue
        xi  = mp.mpf(xf)
        jlv  = _jl(l,   qv*xi)
        jl1v = _jl(l+1, qv*xi)
        jl2v = _jl(l+2, qv*xi)
        U[i]   = complex(Cv * xi * jlv  + Pi0v * xi**(lv+1))
        dU[i]  = complex(Cv * ((lv+1)*jlv - qv*xi*jl1v) + Pi0v*(lv+1)*xi**lv)
        d2U[i] = complex(Cv * (lv*(lv+1)/xi*jlv - (2*lv+3)*qv*jl1v + qv**2*xi*jl2v)
                         + Pi0v*lv*(lv+1)*xi**(lv-1))
    return U, dU, d2U

def _legendre_arrays(l, u_pts):
    Plv  = np.array([float(mp.legendre(l,   float(u))) for u in u_pts])
    Pl1v = np.array([float(mp.legendre(l-1, float(u))) for u in u_pts])
    with np.errstate(divide='ignore', invalid='ignore'):
        dPlv  = np.where(np.abs(u_pts) < 1-1e-14, l*(Pl1v - u_pts*Plv)/(1 - u_pts**2), 0.0)
        d2Plv = np.where(np.abs(u_pts) < 1-1e-14, (2*u_pts*dPlv - l*(l+1)*Plv)/(1 - u_pts**2), 0.0)
    return Plv, dPlv, d2Plv, np.sqrt(np.maximum(1 - u_pts**2, 0))

def gdot_sq_matrix(f_arr, df_arr, x_arr, l, u_pts, d2f_arr):
    g_arr  = (2*f_arr + x_arr*df_arr) / (l*(l+1))
    dg_arr = (3*df_arr + x_arr*d2f_arr) / (l*(l+1))
    Plv, dPlv, d2Plv, sin_th = _legendre_arrays(l, u_pts)
    H = np.zeros((len(x_arr), len(u_pts)))
    for i, xv in enumerate(x_arr):
        if xv < 1e-12: continue
        f_ = f_arr[i]; fp = df_arr[i]; g_ = g_arr[i]; gp = dg_arr[i]
        e_rr   =  fp * Plv
        e_thth = (f_/xv)*Plv - (g_/xv)*(u_pts*dPlv - sin_th**2*d2Plv)
        e_phph = (f_/xv)*Plv - (g_/xv)*(u_pts*dPlv)
        e_rth  =  0.5*sin_th*dPlv*(g_/xv - gp - f_/xv)
        H[i]   = 2*(e_rr**2 + e_thth**2 + e_phph**2 + 2*e_rth**2)
    return H

def gdot_cross_matrix(f1, df1, d2f1, f2, df2, d2f2, x_arr, l, u_pts):
    g1  = (2*f1 + x_arr*df1)  / (l*(l+1)); dg1 = (3*df1 + x_arr*d2f1) / (l*(l+1))
    g2  = (2*f2 + x_arr*df2)  / (l*(l+1)); dg2 = (3*df2 + x_arr*d2f2) / (l*(l+1))
    Plv, dPlv, d2Plv, sin_th = _legendre_arrays(l, u_pts)
    Hc = np.zeros((len(x_arr), len(u_pts)))
    for i, xv in enumerate(x_arr):
        if xv < 1e-12: continue
        f1_=f1[i]; fp1=df1[i]; g1_=g1[i]; gp1=dg1[i]
        f2_=f2[i]; fp2=df2[i]; g2_=g2[i]; gp2=dg2[i]
        e_rr1   =  fp1 * Plv
        e_thth1 = (f1_/xv)*Plv - (g1_/xv)*(u_pts*dPlv - sin_th**2*d2Plv)
        e_phph1 = (f1_/xv)*Plv - (g1_/xv)*(u_pts*dPlv)
        e_rth1  =  0.5*sin_th*dPlv*(g1_/xv - gp1 - f1_/xv)
        e_rr2   =  fp2 * Plv
        e_thth2 = (f2_/xv)*Plv - (g2_/xv)*(u_pts*dPlv - sin_th**2*d2Plv)
        e_phph2 = (f2_/xv)*Plv - (g2_/xv)*(u_pts*dPlv)
        e_rth2  =  0.5*sin_th*dPlv*(g2_/xv - gp2 - f2_/xv)
        Hc[i] = 2*(e_rr1*e_rr2 + e_thth1*e_thth2 + e_phph1*e_phph2 + 2*e_rth1*e_rth2)
    return Hc

print("Helpers loaded (unchanged from the old notebook -- rheology-independent Newtonian machinery):")
print("find_eigenvalue, eigenfunction (U,dU,d2U), gdot_sq_matrix, gdot_cross_matrix")


In [ ]:
def norm_p(p):
    '''<sin^(2p)> over a full period, closed form (Wallis/Gamma). At p=2 this is
    the old notebook's 3/8 exactly -- verified below.'''
    return mp.gamma(p+0.5) / (mp.sqrt(mp.pi) * mp.gamma(p+1))

def compute_gamma_l_a(Oh_val, a_val, l=2, n_x=300, n_theta=100, n_phi=128):
    '''Generalized finite-Oh Gamma_l^(a)(Oh) via direct numerical time-averaging.
    Reduces EXACTLY to the old notebook's compute_gamma_l at a=2 (verified below).'''
    qv = find_eigenvalue(Oh_val, l=l)
    sigma = qv**2 * mp.mpf(Oh_val)
    gamma_ = float(mp.re(sigma)); omega_ = -float(mp.im(sigma))

    x_arr = np.linspace(1e-4, 1.0, n_x)
    U, dU, d2U = eigenfunction(x_arr, qv, l=l)
    u_pts, wts = leggauss(n_theta)
    fR, dfR, d2fR = U.real, dU.real, d2U.real
    fI, dfI, d2fI = U.imag, dU.imag, d2U.imag
    H_R = gdot_sq_matrix(fR, dfR, x_arr, l, u_pts, d2fR)
    H_I = gdot_sq_matrix(fI, dfI, x_arr, l, u_pts, d2fI)
    H_cross = gdot_cross_matrix(fR, dfR, d2fR, fI, dfI, d2fI, x_arr, l, u_pts)

    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    p = (a_val + 2) / 2
    time_avg = np.zeros_like(H_R)
    for phv in phi:
        sinp, cosp = np.sin(phv), np.cos(phv)
        bracket = H_R*sinp**2 + H_I*cosp**2 + 2*H_cross*sinp*cosp
        time_avg += np.abs(bracket)**p
    time_avg /= n_phi
    time_avg /= float(norm_p(p))   # generalizes the old "divide by 3/8" convention

    numerator = np.trapezoid((time_avg @ wts) * x_arr**2, x_arr)
    N_l = np.trapezoid((U.real**2 + U.imag**2) * x_arr**2, x_arr)
    return numerator / N_l**p, gamma_, omega_

# ASSERTION 26: at a=2, MUST reproduce the OLD notebook's compute_gamma_l EXACTLY
# (not just approximately) -- the crux check for this whole section's generalization.
def compute_gamma_l_original(Oh_val, l=2, n_x=300, n_theta=100):
    qv = find_eigenvalue(Oh_val, l=l)
    sigma = qv**2 * mp.mpf(Oh_val)
    gamma_ = float(mp.re(sigma)); omega_ = -float(mp.im(sigma))
    x_arr = np.linspace(1e-4, 1.0, n_x)
    U, dU, d2U = eigenfunction(x_arr, qv, l=l)
    u_pts, wts = leggauss(n_theta)
    H_R = gdot_sq_matrix(U.real, dU.real, x_arr, l, u_pts, d2U.real)
    H_I = gdot_sq_matrix(U.imag, dU.imag, x_arr, l, u_pts, d2U.imag)
    H_cross = gdot_cross_matrix(U.real, dU.real, d2U.real, U.imag, dU.imag, d2U.imag, x_arr, l, u_pts)
    H_sq = H_R**2 + H_I**2 + (2/3)*H_R*H_I + (4/3)*H_cross**2
    numerator = np.trapezoid((H_sq @ wts) * x_arr**2, x_arr)
    N_l = np.trapezoid((U.real**2 + U.imag**2) * x_arr**2, x_arr)
    return numerator / N_l**2, gamma_, omega_

print("Comparing generalized (numerical time-average) vs. old (closed-form) at a=2:")
mismatches = []
for Oh_v in [0.001, 0.05, 0.1, 0.3]:
    G_gen, gam_gen, om_gen = compute_gamma_l_a(Oh_v, 2.0, l=2)
    G_orig, gam_orig, om_orig = compute_gamma_l_original(Oh_v, l=2)
    rel_err = abs(G_gen - G_orig) / G_orig
    print(f"Oh={Oh_v}: generalized={G_gen:.6f}  original={G_orig:.6f}  rel_err={rel_err:.2e}")
    if rel_err > 1e-8:
        mismatches.append(Oh_v)
assert not mismatches
print("ASSERTION 26 OK: generalized finite-Oh Gamma_l^(a) matches the old notebook's")
print("compute_gamma_l EXACTLY (to ~1e-13 relative) at a=2, for every Oh tested")

# ASSERTION 27: norm_p(2) = 3/8 exactly, confirming the generalized normalization
# collapses to the old notebook's specific "divide by 3/8" convention at a=2.
assert abs(float(norm_p(2)) - 0.375) < 1e-12
print("ASSERTION 27 OK: norm_p(2) = 3/8 exactly, matching the old notebook's convention")


In [ ]:
Gamma_inv_ref = 1783566 / 385   # exact symbolic value from Section 6

print(f"{'Oh':>6}  {'a=1.0':>10}  {'a=2.0':>10}  {'a=3.0':>10}")
print("-" * 44)
_sweep_results_a2 = []
for Oh_v in [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5]:
    row = [Oh_v]
    for a_val in (1.0, 2.0, 3.0):
        G, gam, om = compute_gamma_l_a(Oh_v, a_val, l=2)
        row.append(G)
        if a_val == 2.0:
            _sweep_results_a2.append((Oh_v, G, gam, om))
    print(f"{row[0]:>6.3f}  {row[1]:>10.2f}  {row[2]:>10.2f}  {row[3]:>10.2f}")
print(f"{'inv (a=2)':>6}  {'':>10}  {Gamma_inv_ref:>10.2f}")

# ASSERTION 28: Gamma_l^(a)(Oh) positive and strictly decreasing with Oh, for EVERY a
# tested -- generalizes the old notebook's ASSERTION 17a/17b to the general-a case.
for a_val in (1.0, 2.0, 3.0):
    vals = [compute_gamma_l_a(Oh_v, a_val, l=2)[0] for Oh_v in [0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5]]
    assert all(v > 0 for v in vals), f"a={a_val}: negative Gamma_l^(a) found"
    assert all(vals[i] > vals[i+1] for i in range(len(vals)-1)), f"a={a_val}: not monotone decreasing"
print()
print("ASSERTION 28 OK: Gamma_l^(a)(Oh) positive and strictly decreasing with Oh, for a in {1,2,3}")

# ASSERTION 29 (= old ASSERTION 17d/17e at a=2, unchanged tolerance bands):
G_001 = _sweep_results_a2[0][1]
rel_corr_001 = (Gamma_inv_ref - G_001) / Gamma_inv_ref
assert 0.04 < rel_corr_001 < 0.12, f"Oh=0.001 correction {rel_corr_001:.4f} outside (0.04,0.12)"
print(f"ASSERTION 29 OK: Oh=0.001 viscous correction (a=2) = {rel_corr_001:.4f}, in the same")
print("  (4%-12%) band the old notebook's ASSERTION 17d established")

gamma_03 = next(r[2] for r in _sweep_results_a2 if abs(r[0]-0.3) < 1e-9)
gamma_Lamb = (2-1)*(2*2+1)*0.3
rel_diff_Lamb = abs(gamma_03 - gamma_Lamb) / gamma_Lamb
assert rel_diff_Lamb < 0.30
print(f"ASSERTION 30 OK: gamma(Oh=0.3, a=2) = {gamma_03:.4f} within 30% of Lamb ({gamma_Lamb:.4f}),")
print("  matching the old notebook's ASSERTION 17e band")


---
## Summary

| # | Statement | Status |
|---|-----------|--------|
| 1 | Leading-order viscosity expansion verified against sympy's own `series()` at $a=2,3,4$; all intermediate powers vanish | ✓ |
| 2-3 | $n=1$ or $\lambda_c=0$ recover Newtonian, for any $a$ | ✓ |
| 4 | Exact reduction to the old notebook's $O(\epsilon^2)$ correction at $a=2$ | ✓ |
| 5-11 | Reid ODE, boundary conditions, incompressibility, Legendre integrals (unchanged, rheology-independent) | ✓ |
| 12-14 | Inviscid limit, exact $x$-factorization, $H(\theta)=3\cos^4\theta+11\cos^2\theta+13$ (unchanged) | ✓ |
| 15 | $\Gamma_l^{(a)}$ reduces EXACTLY to the old notebook's sympy-exact $\Gamma_2=1783566/385$ at $a=2$ | ✓ |
| 16 | Generalized dissipative force $\propto\vert\dot b\vert^a\dot b$ verified numerically | ✓ |
| 17 | Generalized $D_{\rm eff}$ reduces exactly to the old notebook's $D_{\rm eff}$ at $a=2$ | ✓ |
| 18 | $\varepsilon_{ST}=0$ recovers Newtonian damping, any $a$ | ✓ |
| 19-21 | Generalized secular factor $C(a)$ matches direct quadrature; $C(2)=3/4$ exactly, confirmed against sympy's own explicit period integral | ✓ |
| 22 | **Generalized slow-amplitude equation collapses EXACTLY to the old boxed equation at $a=2$** | ✓ |
| 23 | $\varepsilon_{ST}=0$ gives Newtonian exponential decay, any $a$ | ✓ |
| 24-25 | General-$l$ $\Gamma_l^{(a)}$ matches the exact $l=2,a=2$ value; positive for $l=2..5$, $a\in\{0.5,1,1.5,2,3\}$ | ✓ |
| 26 | **Generalized finite-Oh $\Gamma_l^{(a)}(\mathrm{Oh})$ matches the old notebook's `compute_gamma_l` to ~$10^{-13}$ relative, at every Oh tested** | ✓ |
| 27 | Generalized time-average normalization $\langle\sin^{a+2}\rangle$ collapses to the old notebook's $3/8$ at $a=2$ | ✓ |
| 28 | $\Gamma_l^{(a)}(\mathrm{Oh})$ positive, strictly decreasing with Oh, for $a\in\{1,2,3\}$ | ✓ |
| 29-30 | Oh=0.001 correction band and Oh=0.3 Lamb-comparison band, unchanged from the old notebook, still hold at $a=2$ | ✓ |

**This notebook is a strict generalization, not a replacement.** Every place
the old, Carreau-only derivation could be checked against this one at $a=2$,
the match is exact (symbolic equality where sympy allows it, ~$10^{-13}$–$10^{-16}$
relative numerically otherwise) — nothing about the validated Carreau physics
changed; the shape exponent $a$ was simply never fixed to begin with.

**Implemented in this same round** (see `julia/src/st_extension.jl` and
`julia/test/test_carreau.jl`): `STParams` gains an `a::Float64` field
(default `2.0`, exactly backward-compatible with every existing Carreau call
site), and the two places the old code hardcoded the exponent 2
(`Gamma_eff = Gamma .* (lambda_c.*sigma0).^2` and
`shear_sq_lag = sum(Gamma_eff.*Adot_prev.^2)`) now use `.^a` (with `abs()`
added to the second, since `a` need not be an even integer).

**Ready for the next step:** once real experimental $(K, m, \mu_0, \mu_\infty)$
Cross-model characterization data is available, converting to Carreau-Yasuda
($\lambda_c=K$, $a=m$, $n=1-m$, $\varepsilon_{ST}=[(\mu_0-\mu_\infty)/\mu_0](1-n)/a$
— §1's note on $\mu_\infty$) and computing $\Gamma_l^{(a)}(\mathrm{Oh})$ for the
specific $a$ via this notebook's §6.5 machinery gives everything
`STParams`/`build_residual_st!` needs to run that fluid.
